# NumPy for Machine Learning
Companion to the NumPy appendix. Each section matches a section of the notes; run the cells in order and change things to see what happens.

## Creating Arrays

In [ ]:
import numpy as np

a = np.array([3, 1, 4, 1, 5])              # 1-D array from a list
M = np.array([[1, 2, 3], [4, 5, 6]])       # 2-D array from a list of lists
print(a.shape, M.shape, M.ndim, M.dtype)

In [ ]:
np.zeros((2, 3))                  # 2 x 3 of zeros (np.ones works the same way)
np.arange(0, 10, 2)               # like range(): start, stop (excluded), step
np.linspace(0, 1, 5)              # 5 evenly spaced values, both ends included
np.eye(3)                         # 3 x 3 identity matrix

rng = np.random.default_rng(42)   # a random generator with a fixed seed
rng.integers(0, 10, size=5)       # random integers in [0, 10)
rng.normal(0, 1, size=3)          # draws from a normal distribution, mean 0, sd 1

## Indexing and Slicing

In [ ]:
a[0], a[-1], a[1:4]      # first, last, positions 1, 2, 3
M[1, 2]                  # row 1, column 2
M[:, 1]                  # every row, column 1: a whole column
M[0]                     # row 0 (same as M[0, :])

## Reshaping

In [ ]:
v = np.arange(12)
v.reshape(3, 4)          # same 12 numbers, now 3 rows x 4 columns
v.reshape(-1, 6).shape   # -1: "work out this dimension for me"
v.reshape(3, 4).ravel()  # back to 1-D

## Aggregating Along Rows or Columns: `axis

In [ ]:
S = np.array([[10, 20, 30],
              [40, 50, 60]])
S.sum()          # everything
S.sum(axis=0)    # collapse the rows: one result per column
S.sum(axis=1)    # collapse the columns: one result per row
S.mean(axis=0)   # column means

## Arithmetic, Comparisons and a Few Workhorses

In [ ]:
x = np.array([1.0, 4.0, 9.0])
np.sqrt(x), x * 2 + 1, np.log(x)

In [ ]:
scores = np.array([0.15, 0.72, 0.45, 0.88, 0.31])
np.where(scores >= 0.5, "yes", "no")     # pick between two values element-wise
scores.argmax()                          # position of the largest value
np.argsort(scores)                       # positions that would sort the array
np.sort(scores)[::-1]                    # sorted, largest first
np.unique(["a", "b", "a", "c", "a"], return_counts=True)

In [ ]:
p, q = np.array([1, 2]), np.array([3, 4])
np.vstack([p, q])             # stack as rows    -> [[1 2] [3 4]]
np.hstack([p, q])             # join end to end  -> [1 2 3 4]
np.column_stack([p, q])       # stack as columns -> [[1 3] [2 4]]
np.concatenate([M, M], axis=0).shape, np.concatenate([M, M], axis=1).shape   # (4, 3) (2, 6)

## Vectorization: Let NumPy Do the Looping

In [ ]:
import time

x = np.random.randn(1000000)
y = np.random.randn(1000000)

# Approach 1: Python Loop
start = time.time()
dist_loop = sum((a - b)**2 for a, b in zip(x, y))**0.5
loop_time = time.time() - start

# Approach 2: Vectorized NumPy
start = time.time()
dist_np = np.sqrt(np.sum((x - y)**2))
np_time = time.time() - start

print(f"Loop time:  {loop_time:.4f} s")
print(f"NumPy time: {np_time:.4f} s")
print(f"Speedup:    {loop_time / np_time:.1f}x")

## Broadcasting

In [ ]:
X = np.random.randn(1000, 10)  # Shape: (1000, 10)
mu = np.mean(X, axis=0)        # Shape: (10,)

# Broadcasting alignment:
# X:   1000 x 10
# mu:     1 x 10  (prepended with 1 on the left)
# Result: 1000 x 10
X_centered = X - mu

print(X_centered.mean(axis=0).round(12))

In [ ]:
X = np.random.randn(1000, 10)
row_means = np.mean(X, axis=1)  # Shape: (1000,)

# Attempting X - row_means directly fails!
# X:         1000 x 10
# row_means:    1 x 1000  (trailing dimensions 10 vs 1000 do not match!)

# Solution: Use np.newaxis to introduce an explicit column dimension
row_means_col = row_means[:, np.newaxis]  # Shape: (1000, 1)

# X:             1000 x 10
# row_means_col: 1000 x 1
# Compatible!
X_row_centered = X - row_means_col

print(X_row_centered.shape)

## Selecting with Conditions and with Index Arrays

In [ ]:
# Predicted probabilities for 5 samples
y_prob = np.array([0.15, 0.72, 0.45, 0.88, 0.31])
cutoff = 0.50

# Boolean mask: array([False,  True, False,  True, False])
is_positive = (y_prob >= cutoff)

# Map booleans to integer class predictions (0 or 1)
y_pred = is_positive.astype(int)
print(y_pred)  # [0, 1, 0, 1, 0]

In [ ]:
X = np.random.randn(100, 5)
y = np.random.randint(0, 2, 100)

# Generate a random permutation of index integers
indices = np.random.permutation(len(X))

# Fancy indexing simultaneously reorders both X and y identically
X_shuffled = X[indices]
y_shuffled = y[indices]

# Extract a mini-batch of size 32
batch_size = 32
X_batch = X_shuffled[:batch_size]
y_batch = y_shuffled[:batch_size]

## Linear Algebra with `np.linalg

In [ ]:
X = np.c_[np.ones(100), np.random.randn(100, 3)]
y = X @ np.array([1.0, 2.0, -1.0, 0.5]) + 0.1 * np.random.randn(100)
# Mathematically correct, but numerically dangerous!
w = np.linalg.inv(X.T @ X) @ (X.T @ y)

In [ ]:
# Recommended: Numerically stable and much faster
w = np.linalg.solve(X.T @ X, X.T @ y)

# Even better: Direct least-squares solver via SVD (handles singular matrices)
w, residuals, rank, s = np.linalg.lstsq(X, y, rcond=None)

print(w)

## Under the Hood: Why NumPy Is Fast

In [ ]:
A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])
row_view = A[0, :]          # a view: shares memory with A
row_view[0] = 999.0
print(A[0, 0])              # 999.0: the original changed
print(A.strides, A.T.strides)   # transposing just swaps the strides

## Try it yourself
1. Standardise every column of a random 5x4 array in one line; check the column means are ~0.
2. Label `[0.15, 0.72, 0.45, 0.88, 0.31]` as buy/skip at a 0.4 cutoff with `np.where`.
3. Print names in decreasing order of score using `argsort`.
4. Predict the shapes: `np.ones((4,1)) + np.ones(3)`, and `np.ones((4,2)) + np.ones(4)`.
5. Rerun the timing with 10,000 and 10,000,000 elements.